# 01 - EfficientNet-B0 Experiments

This notebook trains and evaluates **EfficientNet-B0** for Task A of the competition: real vs synthetic image classification.

EfficientNet-B0 is a lightweight CNN model. It is fast, stable, and useful as our first strong baseline. Since it trains faster than heavier models, we can use it to test ideas like learning rate changes, batch size changes, augmentation types, and dataset choices before trying them on larger models.

## What this notebook does

- Loads the selected training datasets from Kaggle.
- Keeps the evaluation dataset fixed so results can be compared fairly across models.
- Trains EfficientNet-B0 using binary classification.
- Predicts whether each image is real or synthetic.
- Calculates accuracy, precision, recall, F1, ROC AUC, Average Precision, EER, and confusion matrix values.
- Saves:
  - `summary.csv` style experiment log
  - `predictions.csv`
  - `training_history.csv`
  - `config.json`
  - `best_model.pth`
  - `kaggle_working.zip`

## Label meaning

- `0 = real`
- `1 = synthetic`

## Important rule

Only change the parameter block at the top before each run. Do not change the evaluation dataset unless the whole team agrees, because all models must be tested on the same evaluation data.

In [1]:
# Install common packages needed for EfficientNet-B0 training and experiment logging
# Note: torch and torchvision are already available in Kaggle GPU notebooks, so we avoid reinstalling them
%pip install numpy pandas scikit-learn matplotlib pillow tqdm


Note: you may need to restart the kernel to use updated packages.


# Imports

In [2]:
# ============================================================
# 3. IMPORTS
# ============================================================
# These imports are shared across most model notebooks.

import os
import json
import random
import shutil
import zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve
)

import torch

print("Imports completed successfully.")

# ============================================================
# 15. EFFICIENTNET-B0 IMPORTS
# ============================================================
# These imports are specific to this model notebook.

from tqdm.auto import tqdm

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights

print("EfficientNet-specific imports completed.")


Imports completed successfully.
EfficientNet-specific imports completed.


In [3]:
# ============================================================
# 4. SEED AND DEVICE SETUP
# ============================================================
# Setting seeds helps make experiments more reproducible.
# Some GPU operations may still have small randomness, but this reduces variation.

def set_seed(seed):
    """
    Set random seeds for Python, NumPy, and PyTorch.

    Parameters:
        seed (int): The seed value used for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Makes PyTorch more deterministic.
    # This can sometimes make training slightly slower, but results become more stable.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Reproducibility
SEED = 42

set_seed(SEED)

# Use GPU if Kaggle gives CUDA access, otherwise use CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Seed set to: {SEED}")
print(f"Using device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("No GPU detected. Make sure Kaggle accelerator is turned on if training.")


Seed set to: 42
Using device: cuda
GPU name: Tesla T4
Number of GPUs: 2


# Variables
CHANGE THIS ONLYYY

In [ ]:
# ============================================================
# 1. SHARED PARAMETER BLOCK
# ============================================================
# Change these values before each experiment run.
# All important experiment settings should stay here so nobody has to search inside the code.

RUNNER_NAME = "zuha"                       # Change to: "zuha", "izma", or "fatima"
MODEL_NAME = "01_efficientnet_b0"             # This notebook is for EfficientNet-B0 experiments
RUN_TYPE = "constrained"                   # Use "constrained" or "open"

# General training settings
EPOCHS = 8
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
OPTIMIZER_NAME = "adamw"                   # Options: "adam", "adamw", "sgd"
WEIGHT_DECAY = 1e-4
SCHEDULER_NAME = "cosine"                    # Options: "none", "cosine", "step"
LOSS_FUNCTION_NAME = "BCEWithLogitsLoss"

# Image/model settings
IMAGE_SIZE = 224
PRETRAINED = True
FREEZE_BACKBONE = False
AUGMENTATION_TYPE = "light_aug"                # Options: "basic", "light_aug", "jpeg_like", "strong"

# Reproducibility
SEED = 42

# Binary classification settings
# Label convention:
# 0 = real
# 1 = synthetic
THRESHOLD = 0.5

# Notes for this experiment.
EXPERIMENT_NOTES = "Experiment #6 = Exp #2 + light_aug augmentation + cosine scheduler + 8 epochs + 1e-3 LR"

print("Shared parameters loaded.")
print(f"Runner: {RUNNER_NAME}")
print(f"Model: {MODEL_NAME}")
print(f"Run type: {RUN_TYPE}")

# ============================================================
# 16. EFFICIENTNET-B0 EXTRA PARAMETERS
# ============================================================
# These are extra settings for the actual training loop.

NUM_WORKERS = 2                  # Kaggle usually works well with 2
PIN_MEMORY = True                # Speeds up GPU loading if CUDA is available
USE_AMP = True                   # Mixed precision training, faster on GPU
GRAD_CLIP_NORM = None            # Example: set 1.0 if gradients become unstable

# Best model selection.
# For this competition, F1 matters most, so we save the model with best validation F1.
SAVE_BEST_BY = "f1"              # Options: "f1" or "val_loss"

# To avoid training/evaluation leakage, remove eval image paths from train_df.
REMOVE_EVAL_FROM_TRAIN = True

print("EfficientNet extra parameters loaded.")
print(f"NUM_WORKERS: {NUM_WORKERS}")
print(f"USE_AMP: {USE_AMP}")
print(f"SAVE_BEST_BY: {SAVE_BEST_BY}")


Shared parameters loaded.
Runner: zuha
Model: 01_efficientnet_b0
Run type: constrained
EfficientNet extra parameters loaded.
NUM_WORKERS: 2
USE_AMP: True
SAVE_BEST_BY: f1


In [5]:
# ============================================================
# 2. DATASET SELECTION BLOCK
# ============================================================
# Training datasets can be changed by the runner.
# Evaluation dataset must stay fixed across ALL model notebooks.

# ----------------------------
# Training dataset choices
# ----------------------------

# IMPORTANT:
# Our uploaded Wang dataset does NOT have train/train.
# It only has:
#   cnn_synth_test/cnn_synth_test
#   val/val
#   test/test
#
# So keep USE_WANG_TRAIN=False.
USE_WANG_TRAIN = False

# Use Wang val as the real+synthetic training source for the first baseline.
USE_WANG_VAL_AS_TRAIN = True

# Keep Wang test OFF for now so it remains available for later internal checks.
USE_WANG_TEST_AS_TRAIN = False

# Optional larger Wang source.
# Keep False for first run. Turn True later if we want more Wang training data.
USE_WANG_CNN_SYNTH_TEST_AS_TRAIN = True

# Corvi is synthetic-only latent diffusion data.
USE_CORVI_TRAIN = True

# Open-run extra training data.
USE_DMIMAGEDETECT_TRAIN = False
USE_REALRAISE_TRAIN = False # check experiment#3, we got a better result in experiment#2 without realraise

# ----------------------------
# Fixed evaluation dataset
# ----------------------------

# Fixed eval for fair comparison across EfficientNet, ConvNeXt, ViT/CLIP, etc.
EVALUATION_DATASET_KEY = "dmimagedetect_test"

# Our uploaded DMImageDetect-Test has 4 real folders with 1000 images each,
# so use 4000 real + 4000 synthetic for balanced evaluation.
MAX_EVAL_REAL = 4000
MAX_EVAL_SYNTHETIC = 4000

# Training image limit per selected dataset.
# Wang val will give real+synthetic, Corvi will give synthetic.
MAX_TRAIN_IMAGES = 6000

# Supported image extensions
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".webp", ".bmp"]

# Avoid exact filepath leakage
REMOVE_EVAL_FROM_TRAIN = True

print("Dataset selection loaded.")
print("Training dataset choices:")
print(f"  Wang train: {USE_WANG_TRAIN}")
print(f"  Wang val as train: {USE_WANG_VAL_AS_TRAIN}")
print(f"  Wang test as train: {USE_WANG_TEST_AS_TRAIN}")
print(f"  Wang cnn_synth_test as train: {USE_WANG_CNN_SYNTH_TEST_AS_TRAIN}")
print(f"  Corvi train: {USE_CORVI_TRAIN}")
print(f"  DMImageDetect train: {USE_DMIMAGEDETECT_TRAIN}")
print(f"  RealRAISE train: {USE_REALRAISE_TRAIN}")
print(f"Fixed evaluation dataset key: {EVALUATION_DATASET_KEY}")
print(f"Fixed eval real/synthetic: {MAX_EVAL_REAL}/{MAX_EVAL_SYNTHETIC}")


Dataset selection loaded.
Training dataset choices:
  Wang train: False
  Wang val as train: True
  Wang test as train: False
  Wang cnn_synth_test as train: True
  Corvi train: True
  DMImageDetect train: False
  RealRAISE train: False
Fixed evaluation dataset key: dmimagedetect_test
Fixed eval real/synthetic: 4000/4000


In [6]:
# ============================================================
# 5. OUTPUT PATHS AND EXPERIMENT ID
# ============================================================
# We keep normal filenames for latest outputs.
# The summary CSV appends every run, while other files are overwritten with latest run.

# Timestamp makes each row in the summary CSV uniquely traceable.
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Clean learning rate text for experiment ID.
lr_text = str(LEARNING_RATE).replace(".", "p").replace("-", "m")

# Experiment ID goes inside the summary CSV.
# We are not using it for filenames because we want simple latest-output filenames.
EXPERIMENT_ID = (
    f"{TIMESTAMP}_{RUNNER_NAME}_{MODEL_NAME}_{RUN_TYPE}"
    f"_ep{EPOCHS}_bs{BATCH_SIZE}_lr{lr_text}_{AUGMENTATION_TYPE}"
)

# Kaggle working directory.
WORKING_DIR = Path("/kaggle/working")

# Model-specific output directory.
MODEL_OUTPUT_DIR = WORKING_DIR / "outputs" / MODEL_NAME
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Normal output filenames.
SUMMARY_CSV_PATH = MODEL_OUTPUT_DIR / f"{MODEL_NAME}_summary.csv"
PREDICTIONS_CSV_PATH = MODEL_OUTPUT_DIR / "predictions.csv"
HISTORY_CSV_PATH = MODEL_OUTPUT_DIR / "training_history.csv"
CONFIG_JSON_PATH = MODEL_OUTPUT_DIR / "config.json"
BEST_MODEL_PATH = MODEL_OUTPUT_DIR / "best_model.pth"
ZIP_PATH = MODEL_OUTPUT_DIR / "kaggle_working.zip"

print("Output paths created.")
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Model output directory: {MODEL_OUTPUT_DIR}")
print(f"Summary CSV path: {SUMMARY_CSV_PATH}")


Output paths created.
Experiment ID: 20260514_143232_zuha_01_efficientnet_b0_constrained_ep8_bs32_lr0p0001_jpeg_like
Model output directory: /kaggle/working/outputs/01_efficientnet_b0
Summary CSV path: /kaggle/working/outputs/01_efficientnet_b0/01_efficientnet_b0_summary.csv


# Dataset

## dataset paths

In [7]:
# ============================================================
# 6. KAGGLE DATASET PATH SETUP
# ============================================================
# Add all possible Kaggle input dataset paths here.
# If a path does not exist in a Kaggle notebook, the scanner will simply skip it.

KAGGLE_INPUT_DIR = Path("/kaggle/input/datasets")

DATASET_PATHS = {
    # DMImageDetect datasets
    "dmimagedetect_test": KAGGLE_INPUT_DIR / "izmakhan26926/dmimagedetect-test",
    "dmimagedetect_train": KAGGLE_INPUT_DIR / "izmakhan26926/dmimagedetect-traintest",
    "dmimagedetect_realraise": KAGGLE_INPUT_DIR / "izmakhan26926/dmimagedetect-realraise",

    # Corvi latent diffusion dataset
    "corvi_latent_diffusion": KAGGLE_INPUT_DIR / "izmakhan26926/corvi-latent-diffusion-trainingset",

    # Wang CNNDetection dataset
    "wang_cnndetection": KAGGLE_INPUT_DIR / "zuhaaqib/wang-cnndetection-dataset",

    # Optional weights dataset
    "clipdet_weights": KAGGLE_INPUT_DIR / "izmakhan26926/dmimagedetect-clipdetweights",
}

print("Checking available Kaggle datasets...\n")

for key, path in DATASET_PATHS.items():
    exists = path.exists()
    print(f"{key:30s} -> {path} | exists = {exists}")


Checking available Kaggle datasets...

dmimagedetect_test             -> /kaggle/input/datasets/izmakhan26926/dmimagedetect-test | exists = True
dmimagedetect_train            -> /kaggle/input/datasets/izmakhan26926/dmimagedetect-traintest | exists = True
dmimagedetect_realraise        -> /kaggle/input/datasets/izmakhan26926/dmimagedetect-realraise | exists = True
corvi_latent_diffusion         -> /kaggle/input/datasets/izmakhan26926/corvi-latent-diffusion-trainingset | exists = True
wang_cnndetection              -> /kaggle/input/datasets/zuhaaqib/wang-cnndetection-dataset | exists = True
clipdet_weights                -> /kaggle/input/datasets/izmakhan26926/dmimagedetect-clipdetweights | exists = True


## helper functions to load, build

In [8]:
# ============================================================
# 7. DATASET SCANNING HELPERS
# ============================================================
# These functions create a dataframe of image paths and labels.
# The dataframe format will be shared across all model notebooks.

REAL_KEYWORDS = [
    "real",
    "authentic",
    "human",
    "nature",
    "raise"
]

SYNTHETIC_KEYWORDS = [
    "fake",
    "synthetic",
    "ai",
    "generated",
    "gan",
    "progan",
    "stylegan",
    "stylegan2",
    "biggan",
    "cyclegan",
    "stargan",
    "gaugan",
    "diffusion",
    "latent",
    "ldm",
    "glide",
    "dalle",
    "stable",
    "midjourney"
]


def is_image_file(path):
    """
    Check whether a file path has an image extension.
    """
    return path.suffix.lower() in IMAGE_EXTENSIONS


def infer_label_from_path(path):
    """
    Infer image label from folder/file path.

    Label convention:
        0 = real
        1 = synthetic
    """
    path_text = str(path).lower()

    # Check synthetic keywords first because some paths may contain words like "realistic".
    for keyword in SYNTHETIC_KEYWORDS:
        if keyword in path_text:
            return 1

    for keyword in REAL_KEYWORDS:
        if keyword in path_text:
            return 0

    return None


def infer_generator_from_path(path):
    """
    Try to infer the generator/source type from the image path.
    This is useful for later error analysis.
    """
    path_text = str(path).lower()

    generator_keywords = [
        "progan", "stylegan", "stylegan2", "biggan", "cyclegan", "stargan",
        "gaugan", "diffusion", "latent", "ldm", "glide", "dalle",
        "stable", "midjourney", "real", "raise"
    ]

    for keyword in generator_keywords:
        if keyword in path_text:
            return keyword

    return "unknown"


def scan_dataset(dataset_key, dataset_path, max_images=None):
    """
    Scan one dataset folder and return a dataframe with image paths and labels.

    This version prints progress while scanning so Kaggle does not look stuck.
    """
    print("\n" + "=" * 70)
    print(f"[scan_dataset] START dataset_key = {dataset_key}")
    print(f"[scan_dataset] Path = {dataset_path}")
    print(f"[scan_dataset] max_images = {max_images}")
    print("=" * 70)

    rows = []

    if not dataset_path.exists():
        print(f"[scan_dataset] SKIP: Dataset path does not exist: {dataset_path}")
        return pd.DataFrame(columns=["filepath", "image_name", "label", "source_dataset", "generator"])

    print("[scan_dataset] Path exists. Starting recursive file scan...")

    image_paths = []
    scanned_items = 0

    # Loop instead of list comprehension so progress is visible.
    for p in dataset_path.rglob("*"):
        scanned_items += 1

        if scanned_items % 50000 == 0:
            print(
                f"[scan_dataset] Still scanning {dataset_key}... "
                f"items seen={scanned_items:,}, image files found={len(image_paths):,}"
            )

        if p.is_file() and is_image_file(p):
            image_paths.append(p)

    print(f"[scan_dataset] Finished recursive scan for {dataset_key}.")
    print(f"[scan_dataset] Total filesystem items seen: {scanned_items:,}")
    print(f"[scan_dataset] Total image files found before shuffle/limit: {len(image_paths):,}")

    # Shuffle before limiting so MAX_TRAIN_IMAGES does not only take one folder/class first.
    print(f"[scan_dataset] Shuffling image paths for {dataset_key}...")
    random.Random(SEED).shuffle(image_paths)

    if max_images is not None:
        print(f"[scan_dataset] Applying max_images limit: {max_images}")
        image_paths = image_paths[:max_images]

    print(f"[scan_dataset] Images to label/process after limit: {len(image_paths):,}")
    print(f"[scan_dataset] Starting label inference for {dataset_key}...")

    skipped_unknown = 0

    for idx, img_path in enumerate(image_paths, start=1):
        if idx % 5000 == 0:
            print(
                f"[scan_dataset] Labeling progress for {dataset_key}: "
                f"{idx:,}/{len(image_paths):,}, rows kept={len(rows):,}, skipped={skipped_unknown:,}"
            )

        label = infer_label_from_path(img_path)

        # If label cannot be inferred, skip it.
        if label is None:
            skipped_unknown += 1
            continue

        rows.append({
            "filepath": str(img_path),
            "image_name": img_path.name,
            "label": label,
            "source_dataset": dataset_key,
            "generator": infer_generator_from_path(img_path)
        })

    df = pd.DataFrame(rows)

    print(f"[scan_dataset] DONE dataset_key = {dataset_key}")
    print(f"[scan_dataset] Images kept with labels: {len(df):,}")
    print(f"[scan_dataset] Skipped unknown-label images: {skipped_unknown:,}")

    if len(df) > 0:
        print("[scan_dataset] Label counts:")
        print(df["label"].value_counts().rename(index={0: "real", 1: "synthetic"}))

    return df


def scan_wang_split(dataset_key, dataset_path, split_name, max_images=None):
    """
    Load only one split from the Wang CNNDetection dataset.

    Wang dataset structure:
        train/train/class_name/0_real
        train/train/class_name/1_fake
        val/val/class_name/0_real
        val/val/class_name/1_fake
        test/test/class_name/0_real
        test/test/class_name/1_fake

    Label convention:
        0 = real
        1 = synthetic
    """
    print("\n" + "=" * 70)
    print(f"[wang_split] START dataset_key = {dataset_key}, split = {split_name}")
    print(f"[wang_split] Base path = {dataset_path}")
    print(f"[wang_split] max_images = {max_images}")
    print("=" * 70)

    # Wang has repeated folder names, for example val/val and test/test.
    split_path = dataset_path / split_name / split_name
    print(f"[wang_split] Expected split path = {split_path}")

    if not split_path.exists():
        print(f"[wang_split] SKIP: Wang split path not found: {split_path}")
        return pd.DataFrame(columns=["filepath", "image_name", "label", "source_dataset", "generator"])

    print("[wang_split] Split path exists. Starting recursive file scan...")

    image_paths = []
    scanned_items = 0

    for p in split_path.rglob("*"):
        scanned_items += 1

        if scanned_items % 50000 == 0:
            print(
                f"[wang_split] Still scanning split={split_name}... "
                f"items seen={scanned_items:,}, image files found={len(image_paths):,}"
            )

        if p.is_file() and is_image_file(p):
            image_paths.append(p)

    print(f"[wang_split] Finished recursive scan for split={split_name}.")
    print(f"[wang_split] Total filesystem items seen: {scanned_items:,}")
    print(f"[wang_split] Total image files found before shuffle/limit: {len(image_paths):,}")

    print(f"[wang_split] Shuffling image paths for split={split_name}...")
    random.Random(SEED).shuffle(image_paths)

    if max_images is not None:
        print(f"[wang_split] Applying max_images limit: {max_images}")
        image_paths = image_paths[:max_images]

    print(f"[wang_split] Images to label/process after limit: {len(image_paths):,}")

    rows = []
    skipped_unknown = 0

    for idx, image_path in enumerate(image_paths, start=1):
        if idx % 5000 == 0:
            print(
                f"[wang_split] Labeling progress split={split_name}: "
                f"{idx:,}/{len(image_paths):,}, rows kept={len(rows):,}, skipped={skipped_unknown:,}"
            )

        path_text = str(image_path).lower()

        if "0_real" in path_text:
            label = 0
            generator = "wang_real"
        elif "1_fake" in path_text:
            label = 1
            generator = "wang_fake"
        else:
            skipped_unknown += 1
            continue

        rows.append({
            "filepath": str(image_path),
            "image_name": image_path.name,
            "label": label,
            "source_dataset": f"{dataset_key}_{split_name}",
            "generator": generator
        })

    df = pd.DataFrame(rows)

    print(f"[wang_split] DONE split = {split_name}")
    print(f"[wang_split] Images kept with labels: {len(df):,}")
    print(f"[wang_split] Skipped unknown-label images: {skipped_unknown:,}")

    if len(df) > 0:
        print("[wang_split] Label counts:")
        print(df["label"].value_counts().rename(index={0: "real", 1: "synthetic"}))

    return df


def print_dataset_summary(df, name):
    """
    Print a clean summary for a dataset dataframe.
    """
    print("\n" + "=" * 70)
    print(f"[summary] DATASET SUMMARY: {name}")
    print("=" * 70)

    if df is None or len(df) == 0:
        print("[summary] No images found.")
        return

    print(f"[summary] Total images: {len(df):,}")

    print("\n[summary] Label counts:")
    print(df["label"].value_counts().rename(index={0: "real", 1: "synthetic"}))

    print("\n[summary] Source dataset counts:")
    print(df["source_dataset"].value_counts())

    print("\n[summary] Generator/source counts:")
    print(df["generator"].value_counts().head(20))


def load_dmimagedetect_fake_from_csv(dataset_key, dataset_path, max_images=None):
    """
    Load only the available synthetic images from DMImageDetect train.

    In list_train.csv:
        filename0 = real COCO image, but these files are missing in our Kaggle input.
        filename1 = synthetic image, available inside coco_latent_t2i.
    """
    print("\n" + "=" * 70)
    print(f"[dm_train_csv] START dataset_key = {dataset_key}")
    print(f"[dm_train_csv] Dataset path = {dataset_path}")
    print(f"[dm_train_csv] max_images = {max_images}")
    print("=" * 70)

    csv_path = dataset_path / "train_set" / "list_train.csv"
    print(f"[dm_train_csv] CSV path = {csv_path}")

    if not csv_path.exists():
        print(f"[dm_train_csv] SKIP: CSV not found: {csv_path}")
        return pd.DataFrame(columns=["filepath", "image_name", "label", "source_dataset", "generator"])

    print("[dm_train_csv] Reading CSV...")
    df_csv = pd.read_csv(csv_path)
    print(f"[dm_train_csv] CSV rows loaded: {len(df_csv):,}")

    print("[dm_train_csv] Shuffling CSV rows...")
    df_csv = df_csv.sample(frac=1, random_state=SEED).reset_index(drop=True)

    if max_images is not None:
        print(f"[dm_train_csv] Applying max_images limit: {max_images}")
        df_csv = df_csv.head(max_images)

    rows = []
    missing_files = 0

    print("[dm_train_csv] Checking fake image paths from filename1...")

    for idx, row in df_csv.iterrows():
        if (idx + 1) % 2000 == 0:
            print(
                f"[dm_train_csv] Progress: {idx + 1:,}/{len(df_csv):,}, "
                f"rows kept={len(rows):,}, missing files={missing_files:,}"
            )

        fake_path = dataset_path / "train_set" / row["filename1"]

        if fake_path.exists():
            rows.append({
                "filepath": str(fake_path),
                "image_name": fake_path.name,
                "label": 1,
                "source_dataset": dataset_key,
                "generator": "coco_latent_t2i"
            })
        else:
            missing_files += 1

    df = pd.DataFrame(rows)

    print(f"[dm_train_csv] DONE dataset_key = {dataset_key}")
    print(f"[dm_train_csv] Synthetic images loaded: {len(df):,}")
    print(f"[dm_train_csv] Missing fake files: {missing_files:,}")

    if len(df) > 0:
        print("[dm_train_csv] Label counts:")
        print(df["label"].value_counts().rename(index={0: "real", 1: "synthetic"}))

    return df


def infer_dmimagedetect_test_label(path):
    """
    Infer labels specifically for DMImageDetect-Test.

    Label convention:
        0 = real
        1 = synthetic

    Real folders start with 'real_'.
    All other image folders inside test_set are treated as synthetic.
    """
    parts = [part.lower() for part in path.parts]

    for part in parts:
        if part.startswith("real_"):
            return 0

    return 1


def scan_dmimagedetect_test_balanced(dataset_key, dataset_path, max_real=5000, max_synthetic=5000):
    """
    Load a balanced fixed evaluation set from DMImageDetect-Test.

    This version prints progress while scanning so Kaggle does not look stuck.
    """
    print("\n" + "=" * 70)
    print(f"[dm_test_eval] START dataset_key = {dataset_key}")
    print(f"[dm_test_eval] Dataset path = {dataset_path}")
    print(f"[dm_test_eval] Target real = {max_real:,}")
    print(f"[dm_test_eval] Target synthetic = {max_synthetic:,}")
    print("=" * 70)

    test_path = dataset_path / "test_set"
    print(f"[dm_test_eval] Expected test path = {test_path}")

    if not test_path.exists():
        print(f"[dm_test_eval] SKIP: DMImageDetect test path not found: {test_path}")
        return pd.DataFrame(columns=["filepath", "image_name", "label", "source_dataset", "generator"])

    real_rows = []
    synthetic_rows = []

    print("[dm_test_eval] Starting recursive scan and collecting balanced eval images...")

    scanned_items = 0
    image_files_seen = 0

    # Important: collect rows during scanning so we can stop early
    # when enough real and synthetic samples are found.
    for image_path in test_path.rglob("*"):
        scanned_items += 1

        if scanned_items % 50000 == 0:
            print(
                f"[dm_test_eval] Still scanning... items seen={scanned_items:,}, "
                f"image files seen={image_files_seen:,}, "
                f"real kept={len(real_rows):,}/{max_real:,}, "
                f"synthetic kept={len(synthetic_rows):,}/{max_synthetic:,}"
            )

        if not image_path.is_file() or not is_image_file(image_path):
            continue

        image_files_seen += 1

        label = infer_dmimagedetect_test_label(image_path)

        row = {
            "filepath": str(image_path),
            "image_name": image_path.name,
            "label": label,
            "source_dataset": dataset_key,
            "generator": image_path.parent.name
        }

        if label == 0 and len(real_rows) < max_real:
            real_rows.append(row)

        elif label == 1 and len(synthetic_rows) < max_synthetic:
            synthetic_rows.append(row)

        # Stop early once both classes reached target.
        if len(real_rows) >= max_real and len(synthetic_rows) >= max_synthetic:
            print("[dm_test_eval] Target reached for both classes. Stopping scan early.")
            break

    print("[dm_test_eval] Finished scan.")
    print(f"[dm_test_eval] Total filesystem items seen: {scanned_items:,}")
    print(f"[dm_test_eval] Total image files seen: {image_files_seen:,}")
    print(f"[dm_test_eval] Real kept: {len(real_rows):,}")
    print(f"[dm_test_eval] Synthetic kept: {len(synthetic_rows):,}")

    df = pd.DataFrame(real_rows + synthetic_rows)

    if len(df) > 0:
        print("[dm_test_eval] Shuffling final eval dataframe...")
        df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    print(f"[dm_test_eval] DONE dataset_key = {dataset_key}")
    print(f"[dm_test_eval] Total eval images loaded: {len(df):,}")

    if len(df) > 0:
        print("[dm_test_eval] Label counts:")
        print(df["label"].value_counts().rename(index={0: "real", 1: "synthetic"}))

        print("[dm_test_eval] Top generators:")
        print(df["generator"].value_counts().head(20))

    return df


## load dataset

In [9]:
# ============================================================
# 8. BUILD TRAINING AND EVALUATION DATAFRAMES
# ============================================================
# Training data is selectable.
# Evaluation data is fixed for all models.

print("\n" + "#" * 80)
print("[build_data] START building training and evaluation dataframes")
print("#" * 80)

print("[build_data] Current dataset switches:")
print(f"[build_data] USE_WANG_TRAIN = {USE_WANG_TRAIN}")
print(f"[build_data] USE_WANG_VAL_AS_TRAIN = {USE_WANG_VAL_AS_TRAIN}")
print(f"[build_data] USE_WANG_TEST_AS_TRAIN = {USE_WANG_TEST_AS_TRAIN}")
print(f"[build_data] USE_WANG_CNN_SYNTH_TEST_AS_TRAIN = {USE_WANG_CNN_SYNTH_TEST_AS_TRAIN}")
print(f"[build_data] USE_CORVI_TRAIN = {USE_CORVI_TRAIN}")
print(f"[build_data] USE_DMIMAGEDETECT_TRAIN = {USE_DMIMAGEDETECT_TRAIN}")
print(f"[build_data] USE_REALRAISE_TRAIN = {USE_REALRAISE_TRAIN}")
print(f"[build_data] EVALUATION_DATASET_KEY = {EVALUATION_DATASET_KEY}")
print(f"[build_data] MAX_TRAIN_IMAGES = {MAX_TRAIN_IMAGES}")
print(f"[build_data] MAX_EVAL_REAL = {MAX_EVAL_REAL}")
print(f"[build_data] MAX_EVAL_SYNTHETIC = {MAX_EVAL_SYNTHETIC}")

train_dfs = []

# Add Wang TRAIN split only if available.
# In our uploaded Wang dataset, train/train does not exist, so this is normally False.
if USE_WANG_TRAIN:
    print("\n[build_data] Loading Wang TRAIN split...")
    wang_train_df = scan_wang_split(
        dataset_key="wang_cnndetection",
        dataset_path=DATASET_PATHS["wang_cnndetection"],
        split_name="train",
        max_images=MAX_TRAIN_IMAGES
    )
    print(f"[build_data] Wang TRAIN loaded rows: {len(wang_train_df):,}")
    train_dfs.append(wang_train_df)
else:
    print("\n[build_data] Skipping Wang TRAIN split because USE_WANG_TRAIN=False")

# Add Wang VAL split as training data.
# This gives both real and synthetic images.
if USE_WANG_VAL_AS_TRAIN:
    print("\n[build_data] Loading Wang VAL split...")
    wang_val_df = scan_wang_split(
        dataset_key="wang_cnndetection",
        dataset_path=DATASET_PATHS["wang_cnndetection"],
        split_name="val",
        max_images=MAX_TRAIN_IMAGES
    )
    print(f"[build_data] Wang VAL loaded rows: {len(wang_val_df):,}")
    train_dfs.append(wang_val_df)
else:
    print("\n[build_data] Skipping Wang VAL split because USE_WANG_VAL_AS_TRAIN=False")

# Add Wang TEST split only if explicitly selected.
if USE_WANG_TEST_AS_TRAIN:
    print("\n[build_data] Loading Wang TEST split...")
    wang_test_df = scan_wang_split(
        dataset_key="wang_cnndetection",
        dataset_path=DATASET_PATHS["wang_cnndetection"],
        split_name="test",
        max_images=MAX_TRAIN_IMAGES
    )
    print(f"[build_data] Wang TEST loaded rows: {len(wang_test_df):,}")
    train_dfs.append(wang_test_df)
else:
    print("\n[build_data] Skipping Wang TEST split because USE_WANG_TEST_AS_TRAIN=False")

# Add Wang CNN synth test split only if explicitly selected.
# Folder structure is cnn_synth_test/cnn_synth_test.
if USE_WANG_CNN_SYNTH_TEST_AS_TRAIN:
    print("\n[build_data] Loading Wang CNN_SYNTH_TEST split...")
    wang_cnn_synth_df = scan_wang_split(
        dataset_key="wang_cnndetection",
        dataset_path=DATASET_PATHS["wang_cnndetection"],
        split_name="cnn_synth_test",
        max_images=MAX_TRAIN_IMAGES
    )
    print(f"[build_data] Wang CNN_SYNTH_TEST loaded rows: {len(wang_cnn_synth_df):,}")
    train_dfs.append(wang_cnn_synth_df)
else:
    print("\n[build_data] Skipping Wang CNN_SYNTH_TEST because USE_WANG_CNN_SYNTH_TEST_AS_TRAIN=False")

# Add Corvi synthetic training data.
if USE_CORVI_TRAIN:
    print("\n[build_data] Loading Corvi latent diffusion training data...")
    corvi_df = scan_dataset(
        dataset_key="corvi_latent_diffusion",
        dataset_path=DATASET_PATHS["corvi_latent_diffusion"],
        max_images=MAX_TRAIN_IMAGES
    )
    print(f"[build_data] Corvi loaded rows: {len(corvi_df):,}")
    train_dfs.append(corvi_df)
else:
    print("\n[build_data] Skipping Corvi because USE_CORVI_TRAIN=False")

# Add only available synthetic DMImageDetect train images.
if USE_DMIMAGEDETECT_TRAIN:
    print("\n[build_data] Loading DMImageDetect train fake-only data...")
    dm_train_df = load_dmimagedetect_fake_from_csv(
        dataset_key="dmimagedetect_train_fake_only",
        dataset_path=DATASET_PATHS["dmimagedetect_train"],
        max_images=MAX_TRAIN_IMAGES
    )
    print(f"[build_data] DMImageDetect train fake-only loaded rows: {len(dm_train_df):,}")
    train_dfs.append(dm_train_df)
else:
    print("\n[build_data] Skipping DMImageDetect train because USE_DMIMAGEDETECT_TRAIN=False")

# Add RealRAISE real images.
if USE_REALRAISE_TRAIN:
    print("\n[build_data] Loading RealRAISE real training data...")
    realraise_df = scan_dataset(
        dataset_key="dmimagedetect_realraise",
        dataset_path=DATASET_PATHS["dmimagedetect_realraise"],
        max_images=MAX_TRAIN_IMAGES
    )
    print(f"[build_data] RealRAISE loaded rows: {len(realraise_df):,}")
    train_dfs.append(realraise_df)
else:
    print("\n[build_data] Skipping RealRAISE because USE_REALRAISE_TRAIN=False")

# Combine selected training datasets.
print("\n[build_data] Combining selected training dataframes...")

if len(train_dfs) > 0:
    train_df = pd.concat(train_dfs, ignore_index=True)
    print(f"[build_data] Combined train rows before duplicate removal: {len(train_df):,}")
else:
    print("[build_data] No training datasets selected. Creating empty train dataframe.")
    train_df = pd.DataFrame(columns=["filepath", "image_name", "label", "source_dataset", "generator"])

# Remove duplicate training filepaths.
print("[build_data] Removing duplicate training filepaths...")
before_dedup = len(train_df)
train_df = train_df.drop_duplicates(subset=["filepath"]).reset_index(drop=True)
after_dedup = len(train_df)

print(f"[build_data] Train rows before dedup: {before_dedup:,}")
print(f"[build_data] Train rows after dedup:  {after_dedup:,}")
print(f"[build_data] Duplicate train rows removed: {before_dedup - after_dedup:,}")

# Build fixed balanced evaluation dataframe from DMImageDetect-Test.
print("\n[build_data] Loading fixed balanced evaluation data from DMImageDetect-Test...")
eval_df = scan_dmimagedetect_test_balanced(
    dataset_key=EVALUATION_DATASET_KEY,
    dataset_path=DATASET_PATHS[EVALUATION_DATASET_KEY],
    max_real=MAX_EVAL_REAL,
    max_synthetic=MAX_EVAL_SYNTHETIC
)

print(f"[build_data] Eval rows before duplicate removal: {len(eval_df):,}")

eval_df = eval_df.drop_duplicates(subset=["filepath"]).reset_index(drop=True)

print(f"[build_data] Eval rows after duplicate removal: {len(eval_df):,}")

print("\n[build_data] Printing final dataset summaries...")
print_dataset_summary(train_df, "TRAINING DATA")
print_dataset_summary(eval_df, "FIXED EVALUATION DATA")

print("\n" + "#" * 80)
print("[build_data] DONE building training and evaluation dataframes")
print("#" * 80)



################################################################################
[build_data] START building training and evaluation dataframes
################################################################################
[build_data] Current dataset switches:
[build_data] USE_WANG_TRAIN = False
[build_data] USE_WANG_VAL_AS_TRAIN = True
[build_data] USE_WANG_TEST_AS_TRAIN = False
[build_data] USE_WANG_CNN_SYNTH_TEST_AS_TRAIN = True
[build_data] USE_CORVI_TRAIN = True
[build_data] USE_DMIMAGEDETECT_TRAIN = False
[build_data] USE_REALRAISE_TRAIN = False
[build_data] EVALUATION_DATASET_KEY = dmimagedetect_test
[build_data] MAX_TRAIN_IMAGES = 6000
[build_data] MAX_EVAL_REAL = 4000
[build_data] MAX_EVAL_SYNTHETIC = 4000

[build_data] Skipping Wang TRAIN split because USE_WANG_TRAIN=False

[build_data] Loading Wang VAL split...

[wang_split] START dataset_key = wang_cnndetection, split = val
[wang_split] Base path = /kaggle/input/datasets/zuhaaqib/wang-cnndetection-dataset
[wang_split] m

In [10]:
# ============================================================
# 17. TRAIN/EVALUATION LEAKAGE SAFETY CHECK
# ============================================================
# This makes sure the same exact image path is not used for both training and evaluation.

if REMOVE_EVAL_FROM_TRAIN:
    before_count = len(train_df)

    eval_paths = set(eval_df["filepath"].tolist())
    train_df = train_df[~train_df["filepath"].isin(eval_paths)].reset_index(drop=True)

    after_count = len(train_df)
    removed_count = before_count - after_count

    print("Train/evaluation leakage check completed.")
    print(f"Training images before removal: {before_count}")
    print(f"Training images after removal:  {after_count}")
    print(f"Removed overlapping images:      {removed_count}")
else:
    print("Leakage removal is disabled. Make sure train/eval datasets are separate.")


Train/evaluation leakage check completed.
Training images before removal: 18000
Training images after removal:  18000
Removed overlapping images:      0


In [11]:
# Quick check to see what Kaggle paths actually exist
for item in Path("/kaggle/input").iterdir():
    print(item)


/kaggle/input/datasets


In [12]:
# ============================================================
# 9. OPTIONAL DATASET CHECK
# ============================================================
# This cell helps verify that paths, labels, and sources are correct.

print("Train label counts:")
print(train_df["label"].value_counts().rename(index={0: "real", 1: "synthetic"}))

print("\nEval label counts:")
print(eval_df["label"].value_counts().rename(index={0: "real", 1: "synthetic"}))

print("\nTrain sources:")
print(train_df["source_dataset"].value_counts())

print("\nEval sources:")
print(eval_df["source_dataset"].value_counts())

print("\nEval generator sample:")
print(eval_df["generator"].value_counts().head(20))

print("\nTraining dataframe preview:")
display(train_df.head())

print("\nEvaluation dataframe preview:")
display(eval_df.head())


Train label counts:
label
synthetic    11994
real          6006
Name: count, dtype: int64

Eval label counts:
label
real         4000
synthetic    4000
Name: count, dtype: int64

Train sources:
source_dataset
wang_cnndetection_val               6000
wang_cnndetection_cnn_synth_test    6000
corvi_latent_diffusion              6000
Name: count, dtype: int64

Eval sources:
source_dataset
dmimagedetect_test    8000
Name: count, dtype: int64

Eval generator sample:
generator
real_imagenet_val                            1000
real_coco_valid                              1000
real_ffhq                                    1000
stargan                                      1000
Deepfloyd-IF_Stage_III                       1000
gaugan                                       1000
real_lsun                                    1000
latent-diffusion_text2img_valid               400
GigaGAN_cond_imagenet256                      300
guided-diffusion_noise2image_LSUNhorses       200
latent-diffusion_noise2im

,filepath,image_name,label,source_dataset,generator
0,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,03508.png,0,wang_cnndetection_val,wang_real
1,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,14986.png,1,wang_cnndetection_val,wang_fake
2,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,12320.png,0,wang_cnndetection_val,wang_real
3,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,14645.png,1,wang_cnndetection_val,wang_fake
4,/kaggle/input/datasets/zuhaaqib/wang-cnndetect...,10802.png,1,wang_cnndetection_val,wang_fake



Evaluation dataframe preview:


,filepath,image_name,label,source_dataset,generator
0,/kaggle/input/datasets/izmakhan26926/dmimagede...,ILSVRC2012_val_00004793.JPEG,0,dmimagedetect_test,real_imagenet_val
1,/kaggle/input/datasets/izmakhan26926/dmimagede...,ILSVRC2012_val_00002088.JPEG,0,dmimagedetect_test,real_imagenet_val
2,/kaggle/input/datasets/izmakhan26926/dmimagede...,000000501523.jpg,0,dmimagedetect_test,real_coco_valid
3,/kaggle/input/datasets/izmakhan26926/dmimagede...,002982.png,0,dmimagedetect_test,real_ffhq
4,/kaggle/input/datasets/izmakhan26926/dmimagede...,Black_Hair_imgHQ22201.png,1,dmimagedetect_test,stargan


In [13]:
# ============================================================
# 18. HARD DATASET SAFETY CHECKS
# ============================================================
# Stop early if the dataset setup is wrong.
# This prevents wasting time on a broken training run.

if len(train_df) == 0:
    raise ValueError(
        "Training dataframe is empty. Check Kaggle dataset paths and selected training datasets."
    )

if len(eval_df) == 0:
    raise ValueError(
        "Evaluation dataframe is empty. Check DMImageDetect-Test path and fixed eval loader."
    )

train_labels = set(train_df["label"].unique())
eval_labels = set(eval_df["label"].unique())

if train_labels != {0, 1}:
    raise ValueError(
        f"Training set must contain both classes 0=real and 1=synthetic. Found: {train_labels}"
    )

if eval_labels != {0, 1}:
    raise ValueError(
        f"Evaluation set must contain both classes 0=real and 1=synthetic. Found: {eval_labels}"
    )

eval_real_count = int((eval_df["label"] == 0).sum())
eval_synth_count = int((eval_df["label"] == 1).sum())

if eval_real_count != eval_synth_count:
    raise ValueError(
        f"Evaluation set is not balanced. Real={eval_real_count}, Synthetic={eval_synth_count}"
    )

print("Hard dataset safety checks passed.")


Hard dataset safety checks passed.


# functions

## metric calculation

In [14]:
# ============================================================
# 10. METRIC CALCULATION FUNCTIONS
# ============================================================
# These functions are shared across all models.
# They calculate the competition-relevant metrics for Task A.

def calculate_eer(y_true, y_prob):
    """
    Calculate Equal Error Rate (EER).

    EER is the point where false positive rate and false negative rate are approximately equal.

    Parameters:
        y_true (array-like): True labels, 0 for real and 1 for synthetic.
        y_prob (array-like): Predicted probability for synthetic class.

    Returns:
        float: Equal Error Rate.
    """
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)

    # False negative rate is 1 - true positive rate.
    fnr = 1 - tpr

    # Find the point where FPR and FNR are closest.
    eer_index = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[eer_index] + fnr[eer_index]) / 2

    return float(eer)


def calculate_binary_metrics(y_true, y_prob, threshold=0.5):
    """
    Calculate binary classification metrics.

    Parameters:
        y_true (array-like): True labels, 0 for real and 1 for synthetic.
        y_prob (array-like): Predicted probability for synthetic class.
        threshold (float): Decision threshold.

    Returns:
        dict: Dictionary containing accuracy, precision, recall, F1, AUC, AP, EER, and confusion matrix values.
    """
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob).astype(float)

    # Convert probabilities into predicted labels using threshold.
    y_pred = (y_prob >= threshold).astype(int)

    # Basic metrics.
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    # Some metrics require both classes to exist.
    # If only one class exists, sklearn can throw an error, so we handle that safely.
    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = np.nan

    try:
        avg_precision = average_precision_score(y_true, y_prob)
    except ValueError:
        avg_precision = np.nan

    try:
        eer = calculate_eer(y_true, y_prob)
    except ValueError:
        eer = np.nan

    # Confusion matrix values.
    # Labels are fixed as [0, 1] so output order is always:
    # [[TN, FP],
    #  [FN, TP]]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc) if not np.isnan(roc_auc) else np.nan,
        "average_precision": float(avg_precision) if not np.isnan(avg_precision) else np.nan,
        "eer": float(eer) if not np.isnan(eer) else np.nan,
        "threshold": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }

    return metrics


def find_best_f1_threshold(y_true, y_prob, thresholds=None):
    """
    Find the threshold that gives the best F1 score.

    Parameters:
        y_true (array-like): True labels.
        y_prob (array-like): Predicted probabilities.
        thresholds (list or None): Thresholds to test.

    Returns:
        tuple: best_threshold, best_f1
    """
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.05)

    best_threshold = 0.5
    best_f1 = -1

    for threshold in thresholds:
        y_pred = (np.array(y_prob) >= threshold).astype(int)
        current_f1 = f1_score(y_true, y_pred, zero_division=0)

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_threshold = threshold

    return float(best_threshold), float(best_f1)


print("Metric functions ready.")


Metric functions ready.


## logging

In [15]:
# ============================================================
# 11. SAVING AND LOGGING FUNCTIONS
# ============================================================
# These functions save predictions, training history, config, and summary rows.

def save_config_json(config_path):
    """
    Save the current experiment settings into config.json.

    Parameters:
        config_path (Path): Path where config JSON should be saved.
    """
    config = {
        "experiment_id": EXPERIMENT_ID,
        "timestamp": TIMESTAMP,
        "runner": RUNNER_NAME,
        "model_name": MODEL_NAME,
        "run_type": RUN_TYPE,

        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "optimizer": OPTIMIZER_NAME,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": SCHEDULER_NAME,
        "loss_function": LOSS_FUNCTION_NAME,

        "image_size": IMAGE_SIZE,
        "pretrained": PRETRAINED,
        "freeze_backbone": FREEZE_BACKBONE,
        "augmentation_type": AUGMENTATION_TYPE,
        "seed": SEED,
        "threshold": THRESHOLD,

        "use_wang_train": USE_WANG_TRAIN,
        "use_wang_val_as_train": USE_WANG_VAL_AS_TRAIN,
        "use_wang_test_as_train": USE_WANG_TEST_AS_TRAIN,
        "use_wang_cnn_synth_test_as_train": USE_WANG_CNN_SYNTH_TEST_AS_TRAIN,
        "use_corvi_train": USE_CORVI_TRAIN,
        "use_dmimagedetect_train": USE_DMIMAGEDETECT_TRAIN,
        "use_realraise_train": USE_REALRAISE_TRAIN,
        "evaluation_dataset_key": EVALUATION_DATASET_KEY,

        "max_train_images": MAX_TRAIN_IMAGES,
        "max_eval_real": MAX_EVAL_REAL,
        "max_eval_synthetic": MAX_EVAL_SYNTHETIC,

        "experiment_notes": EXPERIMENT_NOTES
    }

    with open(config_path, "w") as f:
        json.dump(config, f, indent=4)

    print(f"Config saved to: {config_path}")


def save_predictions_csv(eval_dataframe, y_prob, threshold, predictions_path):
    """
    Save latest evaluation predictions to predictions.csv.

    Parameters:
        eval_dataframe (pd.DataFrame): Evaluation dataframe with filepath and true label.
        y_prob (array-like): Predicted probability for synthetic class.
        threshold (float): Decision threshold.
        predictions_path (Path): Output CSV path.

    Returns:
        pd.DataFrame: Predictions dataframe.
    """
    y_prob = np.array(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    pred_df = eval_dataframe.copy()
    pred_df["experiment_id"] = EXPERIMENT_ID
    pred_df["runner"] = RUNNER_NAME
    pred_df["model_name"] = MODEL_NAME
    pred_df["true_label"] = pred_df["label"].astype(int)
    pred_df["pred_prob"] = y_prob
    pred_df["threshold"] = threshold
    pred_df["pred_label"] = y_pred
    pred_df["correct"] = pred_df["true_label"] == pred_df["pred_label"]

    # Reorder important columns first.
    first_cols = [
        "experiment_id",
        "runner",
        "model_name",
        "filepath",
        "image_name",
        "true_label",
        "pred_prob",
        "threshold",
        "pred_label",
        "correct",
        "source_dataset",
        "generator"
    ]

    remaining_cols = [col for col in pred_df.columns if col not in first_cols]
    pred_df = pred_df[first_cols + remaining_cols]

    pred_df.to_csv(predictions_path, index=False)

    print(f"Predictions saved to: {predictions_path}")
    print(f"Total predictions saved: {len(pred_df)}")

    return pred_df


def save_training_history_csv(history, history_path):
    """
    Save latest training history to training_history.csv.

    Parameters:
        history (list of dict): One dictionary per epoch.
        history_path (Path): Output CSV path.

    Returns:
        pd.DataFrame: History dataframe.
    """
    history_df = pd.DataFrame(history)

    if len(history_df) > 0:
        history_df["experiment_id"] = EXPERIMENT_ID
        history_df["runner"] = RUNNER_NAME
        history_df["model_name"] = MODEL_NAME

        # Put important columns first.
        first_cols = ["experiment_id", "runner", "model_name"]
        remaining_cols = [col for col in history_df.columns if col not in first_cols]
        history_df = history_df[first_cols + remaining_cols]

    history_df.to_csv(history_path, index=False)

    print(f"Training history saved to: {history_path}")
    print(f"History rows saved: {len(history_df)}")

    return history_df


def append_summary_row(summary_path, metrics, best_epoch=None, best_val_loss=None):
    """
    Append one row to the model summary CSV.

    Parameters:
        summary_path (Path): Model summary CSV path.
        metrics (dict): Final evaluation metrics.
        best_epoch (int or None): Best epoch number.
        best_val_loss (float or None): Best validation loss.

    Returns:
        pd.DataFrame: Full updated summary dataframe.
    """
    train_real_count = int((train_df["label"] == 0).sum()) if len(train_df) > 0 else 0
    train_synth_count = int((train_df["label"] == 1).sum()) if len(train_df) > 0 else 0
    eval_real_count = int((eval_df["label"] == 0).sum()) if len(eval_df) > 0 else 0
    eval_synth_count = int((eval_df["label"] == 1).sum()) if len(eval_df) > 0 else 0

    train_datasets_used = ",".join(sorted(train_df["source_dataset"].unique())) if len(train_df) > 0 else ""

    row = {
        "experiment_id": EXPERIMENT_ID,
        "timestamp": TIMESTAMP,
        "runner": RUNNER_NAME,
        "model_name": MODEL_NAME,
        "run_type": RUN_TYPE,

        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "optimizer": OPTIMIZER_NAME,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": SCHEDULER_NAME,
        "loss_function": LOSS_FUNCTION_NAME,

        "image_size": IMAGE_SIZE,
        "pretrained": PRETRAINED,
        "freeze_backbone": FREEZE_BACKBONE,
        "augmentation_type": AUGMENTATION_TYPE,
        "seed": SEED,

        "train_datasets_used": train_datasets_used,
        "evaluation_dataset": EVALUATION_DATASET_KEY,
        "use_wang_train": USE_WANG_TRAIN,
        "use_wang_val_as_train": USE_WANG_VAL_AS_TRAIN,
        "use_wang_test_as_train": USE_WANG_TEST_AS_TRAIN,
        "use_wang_cnn_synth_test_as_train": USE_WANG_CNN_SYNTH_TEST_AS_TRAIN,
        "use_corvi_train": USE_CORVI_TRAIN,
        "use_dmimagedetect_train": USE_DMIMAGEDETECT_TRAIN,
        "use_realraise_train": USE_REALRAISE_TRAIN,
        "num_train_images": len(train_df),
        "num_eval_images": len(eval_df),
        "real_train_count": train_real_count,
        "synthetic_train_count": train_synth_count,
        "real_eval_count": eval_real_count,
        "synthetic_eval_count": eval_synth_count,

        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,

        "accuracy": metrics.get("accuracy"),
        "precision": metrics.get("precision"),
        "recall": metrics.get("recall"),
        "f1": metrics.get("f1"),
        "roc_auc": metrics.get("roc_auc"),
        "average_precision": metrics.get("average_precision"),
        "eer": metrics.get("eer"),
        "threshold": metrics.get("threshold"),

        "tn": metrics.get("tn"),
        "fp": metrics.get("fp"),
        "fn": metrics.get("fn"),
        "tp": metrics.get("tp"),

        "notes": EXPERIMENT_NOTES
    }

    row_df = pd.DataFrame([row])

    # If summary CSV already exists, append to it.
    if summary_path.exists():
        old_summary_df = pd.read_csv(summary_path)
        updated_summary_df = pd.concat([old_summary_df, row_df], ignore_index=True)
    else:
        updated_summary_df = row_df

    updated_summary_df.to_csv(summary_path, index=False)

    print(f"Summary row appended to: {summary_path}")
    print(f"Total experiments logged for {MODEL_NAME}: {len(updated_summary_df)}")

    return updated_summary_df


print("Saving and logging functions ready.")


Saving and logging functions ready.


# data preprocessing

## image transform

In [16]:
# ============================================================
# 18. IMAGE TRANSFORMS
# ============================================================
# These transforms prepare images for EfficientNet-B0.
# EfficientNet pretrained weights expect ImageNet-style normalization.

# ImageNet mean and std are used because EfficientNet-B0 was pretrained on ImageNet.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_train_transforms(augmentation_type):
    """
    Create training transforms based on augmentation type.

    Parameters:
        augmentation_type (str): Type of augmentation to use.

    Returns:
        torchvision.transforms.Compose: Training transform pipeline.
    """

    if augmentation_type == "basic":
        # Simple baseline transform.
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    elif augmentation_type == "light_aug":
        # Slightly stronger augmentation.
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(
                brightness=0.15,
                contrast=0.15,
                saturation=0.10,
                hue=0.02
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    elif augmentation_type == "jpeg_like":
        # Approximation of real-world resizing/cropping effects.
        # JPEG compression itself is not added here to keep it simple and fast.
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE + 48, IMAGE_SIZE + 48)),
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=3)
            ], p=0.20),
            transforms.ColorJitter(
                brightness=0.20,
                contrast=0.20,
                saturation=0.15,
                hue=0.02
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    else:
        raise ValueError(f"Unknown AUGMENTATION_TYPE: {augmentation_type}")


def get_eval_transforms():
    """
    Create evaluation transforms.

    Returns:
        torchvision.transforms.Compose: Evaluation transform pipeline.
    """
    return transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


train_transforms = get_train_transforms(AUGMENTATION_TYPE)
eval_transforms = get_eval_transforms()

print("Transforms created.")
print(f"Training augmentation type: {AUGMENTATION_TYPE}")


Transforms created.
Training augmentation type: jpeg_like


## pytorch

In [17]:
# ============================================================
# 19. PYTORCH IMAGE DATASET CLASS
# ============================================================
# This class reads image paths from train_df/eval_df and returns tensors + labels.

class BinaryImageDataset(Dataset):
    """
    Dataset for binary real vs synthetic image classification.

    Each item returns:
        image tensor
        label tensor
        image path
    """

    def __init__(self, dataframe, transform=None):
        """
        Parameters:
            dataframe (pd.DataFrame): Dataframe with filepath and label columns.
            transform: Torchvision transforms.
        """
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        """
        Returns:
            int: Number of images in the dataset.
        """
        return len(self.dataframe)

    def __getitem__(self, idx):
        """
        Load one image and its label.

        Parameters:
            idx (int): Row index.

        Returns:
            image (Tensor): Transformed image.
            label (Tensor): Float label, 0.0 for real and 1.0 for synthetic.
            filepath (str): Original image path.
        """
        row = self.dataframe.iloc[idx]
        image_path = row["filepath"]
        label = float(row["label"])

        # Convert image to RGB so all inputs have 3 channels.
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        # BCEWithLogitsLoss expects float labels.
        label = torch.tensor(label, dtype=torch.float32)

        return image, label, image_path


print("BinaryImageDataset class ready.")


BinaryImageDataset class ready.


## dataloaders

In [18]:
# ============================================================
# 20. CREATE DATASETS AND DATALOADERS
# ============================================================
# DataLoaders feed batches of images into the model.

train_dataset = BinaryImageDataset(train_df, transform=train_transforms)
eval_dataset = BinaryImageDataset(eval_df, transform=eval_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Datasets and DataLoaders created.")
print(f"Training images:   {len(train_dataset)}")
print(f"Evaluation images: {len(eval_dataset)}")
print(f"Training batches:  {len(train_loader)}")
print(f"Evaluation batches:{len(eval_loader)}")


Datasets and DataLoaders created.
Training images:   18000
Evaluation images: 8000
Training batches:  563
Evaluation batches:250


# model

## create model

In [19]:
# ============================================================
# 21. CREATE EFFICIENTNET-B0 MODEL
# ============================================================
# We use pretrained EfficientNet-B0 and replace the final classifier for binary classification.

def create_efficientnet_b0(pretrained=True, freeze_backbone=False):
    """
    Create EfficientNet-B0 for binary classification.

    Parameters:
        pretrained (bool): Whether to load ImageNet pretrained weights.
        freeze_backbone (bool): Whether to freeze feature extractor layers.

    Returns:
        torch.nn.Module: EfficientNet-B0 model.
    """

    if pretrained:
        weights = EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        print("Loaded EfficientNet-B0 with ImageNet pretrained weights.")
    else:
        model = models.efficientnet_b0(weights=None)
        print("Loaded EfficientNet-B0 without pretrained weights.")

    # Freeze backbone if selected.
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
        print("Backbone frozen. Only classifier will train.")
    else:
        print("Backbone is trainable.")

    # EfficientNet-B0 classifier usually looks like:
    # Sequential(
    #   (0): Dropout(...)
    #   (1): Linear(in_features=1280, out_features=1000)
    # )
    in_features = model.classifier[1].in_features

    # Replace final layer with one output logit.
    # One logit is enough for binary classification with BCEWithLogitsLoss.
    model.classifier[1] = nn.Linear(in_features, 1)

    return model


model = create_efficientnet_b0(
    pretrained=PRETRAINED,
    freeze_backbone=FREEZE_BACKBONE
)

model = model.to(DEVICE)

print("Model created and moved to device.")
print(model.classifier)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 149MB/s]


Loaded EfficientNet-B0 with ImageNet pretrained weights.
Backbone is trainable.
Model created and moved to device.
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1, bias=True)
)


## optimizers, schedulers

In [20]:
# ============================================================
# 22. LOSS FUNCTION, OPTIMIZER, AND SCHEDULER
# ============================================================
# This cell creates the training components.

def create_optimizer(model, optimizer_name):
    """
    Create optimizer based on selected optimizer name.

    Parameters:
        model (torch.nn.Module): Model to train.
        optimizer_name (str): Optimizer name.

    Returns:
        torch.optim.Optimizer: Optimizer.
    """

    # Only train parameters where requires_grad=True.
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    if optimizer_name.lower() == "adam":
        optimizer = torch.optim.Adam(
            trainable_params,
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY
        )

    elif optimizer_name.lower() == "adamw":
        optimizer = torch.optim.AdamW(
            trainable_params,
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY
        )

    elif optimizer_name.lower() == "sgd":
        optimizer = torch.optim.SGD(
            trainable_params,
            lr=LEARNING_RATE,
            momentum=0.9,
            weight_decay=WEIGHT_DECAY
        )

    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    return optimizer


def create_scheduler(optimizer, scheduler_name):
    """
    Create learning rate scheduler.

    Parameters:
        optimizer: PyTorch optimizer.
        scheduler_name (str): Scheduler name.

    Returns:
        scheduler or None
    """

    if scheduler_name.lower() == "none":
        return None

    elif scheduler_name.lower() == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=EPOCHS
        )

    elif scheduler_name.lower() == "step":
        return torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=3,
            gamma=0.5
        )

    else:
        raise ValueError(f"Unknown scheduler: {scheduler_name}")


criterion = nn.BCEWithLogitsLoss()
optimizer = create_optimizer(model, OPTIMIZER_NAME)
scheduler = create_scheduler(optimizer, SCHEDULER_NAME)

print("Loss, optimizer, and scheduler created.")
print(f"Loss function: {LOSS_FUNCTION_NAME}")
print(f"Optimizer: {OPTIMIZER_NAME}")
print(f"Scheduler: {SCHEDULER_NAME}")


Loss, optimizer, and scheduler created.
Loss function: BCEWithLogitsLoss
Optimizer: adamw
Scheduler: cosine


# training

## functions

In [21]:
# ============================================================
# 23. TRAINING AND EVALUATION FUNCTIONS
# ============================================================
# These functions handle one training epoch and one evaluation pass.

def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None):
    """
    Train the model for one epoch.

    Parameters:
        model: PyTorch model.
        loader: Training DataLoader.
        criterion: Loss function.
        optimizer: Optimizer.
        device: CUDA or CPU.
        scaler: GradScaler for mixed precision.

    Returns:
        float: Average training loss.
    """

    model.train()
    running_loss = 0.0
    total_samples = 0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for images, labels, paths in progress_bar:
        images = images.to(device)
        labels = labels.to(device).view(-1, 1)

        optimizer.zero_grad()

        # Mixed precision speeds up training on GPU.
        if scaler is not None and device.type == "cuda":
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()

            # Optional gradient clipping.
            if GRAD_CLIP_NORM is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

            scaler.step(optimizer)
            scaler.update()

        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()

            # Optional gradient clipping.
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

            optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

        progress_bar.set_postfix({"loss": loss.item()})

    avg_loss = running_loss / max(total_samples, 1)
    return avg_loss


def evaluate_model(model, loader, criterion, device):
    """
    Evaluate model on validation/evaluation dataset.

    Parameters:
        model: PyTorch model.
        loader: Evaluation DataLoader.
        criterion: Loss function.
        device: CUDA or CPU.

    Returns:
        tuple: avg_loss, y_true, y_prob, image_paths
    """

    model.eval()
    running_loss = 0.0
    total_samples = 0

    all_labels = []
    all_probs = []
    all_paths = []

    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for images, labels, paths in progress_bar:
            images = images.to(device)
            labels = labels.to(device).view(-1, 1)

            logits = model(images)
            loss = criterion(logits, labels)

            # Convert logits to probabilities for synthetic class.
            probs = torch.sigmoid(logits).view(-1)

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

            all_labels.extend(labels.view(-1).cpu().numpy().astype(int).tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
            all_paths.extend(list(paths))

            progress_bar.set_postfix({"loss": loss.item()})

    avg_loss = running_loss / max(total_samples, 1)

    return avg_loss, np.array(all_labels), np.array(all_probs), all_paths


print("Training and evaluation functions ready.")


Training and evaluation functions ready.


## run training

In [22]:
# ============================================================
# 24. MAIN TRAINING LOOP
# ============================================================
# This trains EfficientNet-B0 and saves the best model based on validation F1.

history = []

best_epoch = None
best_val_loss = float("inf")
best_val_f1 = -1.0

# GradScaler is used only when AMP is enabled and CUDA is available.
scaler = torch.cuda.amp.GradScaler() if (USE_AMP and DEVICE.type == "cuda") else None

print("=" * 70)
print("STARTING EFFICIENTNET-B0 TRAINING")
print("=" * 70)
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Training images: {len(train_dataset)}")
print(f"Evaluation images: {len(eval_dataset)}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Optimizer: {OPTIMIZER_NAME}")
print(f"Augmentation: {AUGMENTATION_TYPE}")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")

    # Train for one epoch.
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        scaler=scaler
    )

    # Evaluate after each epoch.
    val_loss, epoch_y_true, epoch_y_prob, _ = evaluate_model(
        model=model,
        loader=eval_loader,
        criterion=criterion,
        device=DEVICE
    )

    # Find best F1 threshold for this epoch.
    epoch_best_threshold, epoch_best_threshold_f1 = find_best_f1_threshold(
        y_true=epoch_y_true,
        y_prob=epoch_y_prob
    )

    # Calculate metrics using the best threshold for this epoch.
    epoch_metrics = calculate_binary_metrics(
        y_true=epoch_y_true,
        y_prob=epoch_y_prob,
        threshold=epoch_best_threshold
    )

    # Step scheduler after validation.
    if scheduler is not None:
        scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    # Save epoch history.
    epoch_row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": epoch_metrics["accuracy"],
        "val_precision": epoch_metrics["precision"],
        "val_recall": epoch_metrics["recall"],
        "val_f1": epoch_metrics["f1"],
        "val_roc_auc": epoch_metrics["roc_auc"],
        "val_average_precision": epoch_metrics["average_precision"],
        "val_eer": epoch_metrics["eer"],
        "best_threshold_for_epoch": epoch_best_threshold,
        "learning_rate": current_lr
    }

    history.append(epoch_row)

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val F1: {epoch_metrics['f1']:.4f} | "
        f"Val Acc: {epoch_metrics['accuracy']:.4f} | "
        f"Val AUC: {epoch_metrics['roc_auc']:.4f} | "
        f"Best Threshold: {epoch_best_threshold:.2f}"
    )

    # Decide if this epoch is the best one.
    if SAVE_BEST_BY == "f1":
        is_best = epoch_metrics["f1"] > best_val_f1
    elif SAVE_BEST_BY == "val_loss":
        is_best = val_loss < best_val_loss
    else:
        raise ValueError(f"Unknown SAVE_BEST_BY: {SAVE_BEST_BY}")

    # Save best model.
    if is_best:
        best_epoch = epoch
        best_val_loss = val_loss
        best_val_f1 = epoch_metrics["f1"]

        torch.save(model.state_dict(), BEST_MODEL_PATH)

        print(f"New best model saved at epoch {epoch}.")
        print(f"Best model path: {BEST_MODEL_PATH}")

print("\nTraining completed.")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best validation F1: {best_val_f1:.4f}")


STARTING EFFICIENTNET-B0 TRAINING
Experiment ID: 20260514_143232_zuha_01_efficientnet_b0_constrained_ep8_bs32_lr0p0001_jpeg_like
Training images: 18000
Evaluation images: 8000
Epochs: 8
Batch size: 32
Learning rate: 0.0001
Optimizer: adamw
Augmentation: jpeg_like

Epoch 1/8


/tmp/ipykernel_57/3177740694.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if (USE_AMP and DEVICE.type == "cuda") else None


Training:   0%|          | 0/563 [00:00<?, ?it/s]

/tmp/ipykernel_57/3222556243.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]

...

Train Loss: 0.0613 | Val Loss: 1.1613 | Val F1: 0.7107 | Val Acc: 0.6405 | Val AUC: 0.7536 | Best Threshold: 0.15
New best model saved at epoch 7.
Best model path: /kaggle/working/outputs/01_efficientnet_b0/best_model.pth

Epoch 8/8


Training:   0%|          | 0/563 [00:00<?, ?it/s]

/tmp/ipykernel_57/3222556243.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]

Train Loss: 0.0554 | Val Loss: 1.2283 | Val F1: 0.7099 | Val Acc: 0.6510 | Val AUC: 0.7489 | Best Threshold: 0.20

Training completed.
Best epoch: 7
Best validation loss: 1.1613
Best validation F1: 0.7107


# evaluation

In [23]:
# ============================================================
# 25. LOAD BEST MODEL AND FINAL EVALUATION
# ============================================================
# We reload the best saved model and evaluate it once more for final logging.

if BEST_MODEL_PATH.exists():
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    model = model.to(DEVICE)
    print(f"Loaded best model from: {BEST_MODEL_PATH}")
else:
    print("WARNING: Best model file not found. Using current model weights.")

final_val_loss, y_true, y_prob, image_paths = evaluate_model(
    model=model,
    loader=eval_loader,
    criterion=criterion,
    device=DEVICE
)

# Find best threshold based on final evaluation F1.
best_threshold, best_threshold_f1 = find_best_f1_threshold(
    y_true=y_true,
    y_prob=y_prob
)

# Calculate final metrics with best threshold.
metrics = calculate_binary_metrics(
    y_true=y_true,
    y_prob=y_prob,
    threshold=best_threshold
)

print("=" * 70)
print("FINAL EVALUATION RESULTS")
print("=" * 70)
print(f"Final validation loss: {final_val_loss:.4f}")
print(f"Best threshold:        {best_threshold:.2f}")
print(f"Accuracy:              {metrics['accuracy']:.4f}")
print(f"Precision:             {metrics['precision']:.4f}")
print(f"Recall:                {metrics['recall']:.4f}")
print(f"F1 Score:              {metrics['f1']:.4f}")
print(f"ROC AUC:               {metrics['roc_auc']:.4f}")
print(f"Average Precision:     {metrics['average_precision']:.4f}")
print(f"EER:                   {metrics['eer']:.4f}")
print(f"TN: {metrics['tn']} | FP: {metrics['fp']} | FN: {metrics['fn']} | TP: {metrics['tp']}")
print("=" * 70)


Loaded best model from: /kaggle/working/outputs/01_efficientnet_b0/best_model.pth


Evaluating:   0%|          | 0/250 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c04351000e0>Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7c04351000e0>Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

    Traceback (most recent call last):
self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        self._shutdown_workers()if w.is_alive():

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     if w.is_alive():  
       ^  ^ ^^^ ^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^^assert self._parent_pid == os.getpid(), 'can only test a child process'

  File "/usr/lib/python

FINAL EVALUATION RESULTS
Final validation loss: 1.1613
Best threshold:        0.15
Accuracy:              0.6405
Precision:             0.5946
Recall:                0.8830
F1 Score:              0.7107
ROC AUC:               0.7536
Average Precision:     0.7623
EER:                   0.3154
TN: 1592 | FP: 2408 | FN: 468 | TP: 3532


# log, save, analyze

In [24]:
# ============================================================
# 26. SAVE OUTPUTS AND APPEND SUMMARY
# ============================================================
# This saves:
# - config.json
# - predictions.csv
# - training_history.csv
# - efficientnet_b0_summary.csv

# Save latest config.
save_config_json(CONFIG_JSON_PATH)

# Save latest predictions.
predictions_df = save_predictions_csv(
    eval_dataframe=eval_df,
    y_prob=y_prob,
    threshold=best_threshold,
    predictions_path=PREDICTIONS_CSV_PATH
)

# Save latest training history.
history_df = save_training_history_csv(
    history=history,
    history_path=HISTORY_CSV_PATH
)

# Append one row to model summary CSV.
summary_df = append_summary_row(
    summary_path=SUMMARY_CSV_PATH,
    metrics=metrics,
    best_epoch=best_epoch,
    best_val_loss=best_val_loss
)

print("\nLatest summary rows:")
display(summary_df.tail())


Config saved to: /kaggle/working/outputs/01_efficientnet_b0/config.json
Predictions saved to: /kaggle/working/outputs/01_efficientnet_b0/predictions.csv
Total predictions saved: 8000
Training history saved to: /kaggle/working/outputs/01_efficientnet_b0/training_history.csv
History rows saved: 8
Summary row appended to: /kaggle/working/outputs/01_efficientnet_b0/01_efficientnet_b0_summary.csv
Total experiments logged for 01_efficientnet_b0: 1

Latest summary rows:


,experiment_id,timestamp,runner,model_name,run_type,epochs,batch_size,learning_rate,optimizer,weight_decay,...,f1,roc_auc,average_precision,eer,threshold,tn,fp,fn,tp,notes
0,20260514_143232_zuha_01_efficientnet_b0_constr...,20260514_143232,zuha,01_efficientnet_b0,constrained,8,32,0.0001,adamw,0.0001,...,0.710664,0.753552,0.762257,0.315375,0.15,1592,2408,468,3532,Exp 4 + jpeg_like augmentation + cosine schedu...


In [25]:
# ============================================================
# 27. QUICK ERROR ANALYSIS
# ============================================================
# This helps us quickly see what the model got wrong.

wrong_df = predictions_df[predictions_df["correct"] == False].copy()

print(f"Total wrong predictions: {len(wrong_df)} out of {len(predictions_df)}")

if len(wrong_df) > 0:
    print("\nWrong prediction counts by true label:")
    print(wrong_df["true_label"].value_counts().rename(index={0: "real", 1: "synthetic"}))

    print("\nWrong prediction counts by generator/source:")
    print(wrong_df["generator"].value_counts().head(20))

    print("\nSample wrong predictions:")
    display(wrong_df[[
        "image_name",
        "true_label",
        "pred_prob",
        "pred_label",
        "source_dataset",
        "generator",
        "filepath"
    ]].head(20))
else:
    print("No wrong predictions found.")


Total wrong predictions: 2876 out of 8000

Wrong prediction counts by true label:
true_label
real         2408
synthetic     468
Name: count, dtype: int64

Wrong prediction counts by generator/source:
generator
real_ffhq                                    905
real_imagenet_val                            773
real_coco_valid                              658
Deepfloyd-IF_Stage_III                       223
guided-diffusion_noise2image_LSUNhorses      196
real_lsun                                     72
GigaGAN_cond_imagenet256                      23
gaugan                                        15
stargan                                        7
latent-diffusion_noise2image_LSUNbedrooms      4
Name: count, dtype: int64

Sample wrong predictions:


,image_name,true_label,pred_prob,pred_label,source_dataset,generator,filepath
0,ILSVRC2012_val_00004793.JPEG,0,0.778260,1,dmimagedetect_test,real_imagenet_val,/kaggle/input/datasets/izmakhan26926/dmimagede...
1,ILSVRC2012_val_00002088.JPEG,0,0.926058,1,dmimagedetect_test,real_imagenet_val,/kaggle/input/datasets/izmakhan26926/dmimagede...
3,002982.png,0,0.910231,1,dmimagedetect_test,real_ffhq,/kaggle/input/datasets/izmakhan26926/dmimagede...
5,ILSVRC2012_val_00001598.JPEG,0,0.981239,1,dmimagedetect_test,real_imagenet_val,/kaggle/input/datasets/izmakhan26926/dmimagede...
7,000000436551.jpg,0,0.983461,1,dmimagedetect_test,real_coco_valid,/kaggle/input/datasets/izmakhan26926/dmimagede...
11,ILSVRC2012_val_00018153.JPEG,0,0.349933,1,dmimagedetect_test,real_imagenet_val,/kaggle/input/datasets/izmakhan26926/dmimagede...
18,ILSVRC2012_val_00000070.JPEG,0,0.999245,1,dmimagedetect_test,real_imagenet_val,/kaggle/input/datasets/izmakhan26926/dmimagede...
19,ILSVRC2012_val_00022278.JPEG,0,0.622133,1,dmimagedetect_test,real_imagenet_val,/kaggle/input/datasets/izmakhan26926/dmimagede...
23,guided-diffusion_noise2img-lsunhorses-nodropou...,1,0.000002,0,dmimagedetect_test,guided-diffusion_noise2image_LSUNhorses,/kaggle/input/datasets/izmakhan26926/dmimagede...
25,001078.png,0,0.996848,1,dmimagedetect_test,real_ffhq,/kaggle/input/datasets/izmakhan26926/dmimagede...


# save outputs in a zip
because the outputs are alot, instead of downloading each manually we can just download one zip and it would download in the correct folder structure and we can just paste in github/vscode

In [26]:
import os
import shutil
from IPython.display import FileLink, display as ipy_display

# Path you want to zip
source_folder = "/kaggle/working"

# Where the zip file will be created
zip_base_path = "/kaggle/working/kaggle_working_backup"

# Create kaggle_working_backup.zip
# shutil.make_archive expects the path without ".zip"
zip_file = shutil.make_archive(zip_base_path, "zip", source_folder)

print("ZIP created at:", zip_file)

# Show a clickable download link inside the notebook
ipy_display(FileLink(zip_file))


ZIP created at: /kaggle/working/kaggle_working_backup.zip


/kaggle/working/kaggle_working_backup.zip